# 🎬 Movie RAG Chatbot — Groq + LLaMA 3.3 70B Edition

## Arsitektur Sistem

```
User Input
    │
    ▼
┌─────────────────┐
│  GUARDRAIL      │  ← Tolak jika bukan topik film
└────────┬────────┘
         │ (on-topic)
         ▼
┌─────────────────┐
│  QUERY ROUTER   │  ← Klasifikasi: metadata / semantic / hybrid
└────────┬────────┘
         │
    ┌────┴────┐
    ▼         ▼
┌───────┐  ┌──────────┐
│Metadata│  │ Semantic │  ← Local ChromaDB Search
│Filter  │  │ Search   │
└───┬───┘  └────┬─────┘
    └────┬───────┘
         │ (0 result / film > 2023)
         ▼
┌─────────────────┐
│  FALLBACK API   │  ← TMDB / OMDb + Sitasi
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│ RESPONSE + MEM  │  ← LLM generate + update history
└─────────────────┘
```

| Modul | Komponen |
|---|---|
| Memory | `ChatMemory` — ConversationBufferWindowMemory manual |
| Guardrail | `TopicGuardrail` — keyword + LLM classifier |
| Router | `QueryRouter` — LLM-based intent extraction |
| Search | `MovieSearchEngine` — metadata filter + semantic + hybrid |
| Fallback | `ExternalMovieAPI` — TMDB / OMDb + sitasi |
| Generator | `ResponseGenerator` — LLM dengan context injection |
| Orchestrator | `MovieRAGChatbot` — pipeline utama |

## ⚙️ Cell 1: Instalasi Dependencies

In [1]:
!pip install -q \
    langchain \
    langchain-core \
    langchain-groq \
    langchain-huggingface \
    langchain-chroma \
    chromadb \
    sentence-transformers \
    pandas numpy \
    requests \
    python-dotenv

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 🔑 Cell 2: Konfigurasi API Keys & Constants

In [14]:
# ============================================================
# CELL 2 — KONFIGURASI
# ============================================================

import os
import json
import re
import requests
import pandas as pd
import numpy as np
from typing import List, Dict, Optional, Tuple, Any
from dataclasses import dataclass, field
from dotenv import load_dotenv

PROJECT_PATH  = '/content/drive/MyDrive/PKA1'

# load_dotenv()  # Load dari file .env jika ada
ENV_PATH      = os.path.join(PROJECT_PATH, 'env')

if os.path.exists(ENV_PATH):
    load_dotenv(ENV_PATH)
    print(f"[OK] .env dimuat dari: {ENV_PATH}")
else:
    print(f"[!] .env tidak ditemukan di {ENV_PATH}")

# ── API Keys ──────────────────────────────────────────────────
# Daftar gratis di: console.groq.com
GROQ_API_KEY    = os.getenv('GROQ_API_KEY',  'gsk_your-groq-key')
TMDB_API_KEY    = os.getenv('TMDB_API_KEY',  '')   # Daftar gratis: themoviedb.org
OMDB_API_KEY    = os.getenv('OMDB_API_KEY',  '')   # Daftar gratis: omdbapi.com

# ── Model Config ──────────────────────────────────────────────
LLM_MODEL           = 'llama-3.3-70b-versatile'  # LLM utama via Groq (gratis, cepat)
EMBEDDING_MODEL     = 'BAAI/bge-small-en-v1.5'   # Embedding lokal gratis (sentence-transformers)
CHROMA_PERSIST_DIR  = os.path.join(PROJECT_PATH, 'chroma_movie_db')       # Lokasi penyimpanan vector store
MOVIE_CSV_PATH      = os.path.join(PROJECT_PATH, 'imdb_movies.csv')              # Path dataset film Anda

# ── Search Config ─────────────────────────────────────────────
DEFAULT_RESULT_LIMIT    = 5   # Default jumlah film yang ditampilkan
MAX_SEMANTIC_CANDIDATES = 20  # Kandidat sebelum re-ranking
MEMORY_WINDOW_TURNS     = 10  # Jumlah turn yang diingat

print('✅ Konfigurasi selesai')
print(f'   LLM         : {LLM_MODEL}')
print(f'   Embedding   : {EMBEDDING_MODEL}')
print(f'   ChromaDB    : {CHROMA_PERSIST_DIR}')
print(f'   TMDB Key    : {"✅ Set" if TMDB_API_KEY else "❌ Belum diset"}')
print(f'   OMDb Key    : {"✅ Set" if OMDB_API_KEY else "❌ Belum diset"}')


[OK] .env dimuat dari: /content/drive/MyDrive/PKA1/env
✅ Konfigurasi selesai
   LLM         : llama-3.3-70b-versatile
   Embedding   : BAAI/bge-small-en-v1.5
   ChromaDB    : /content/drive/MyDrive/PKA1/chroma_movie_db
   TMDB Key    : ❌ Belum diset
   OMDb Key    : ✅ Set


## 📂 Cell 3: Modul Data Loading & Vector Store

> **Apa yang diubah:** Ditambahkan `MovieDataLoader` dengan auto-normalisasi kolom
> agar kompatibel dengan berbagai format CSV (IMDB, TMDB, Kaggle).
> `VectorStoreManager` mengelola ChromaDB dengan metadata terstruktur.

In [15]:
# ============================================================
# CELL 3 — DATA LOADING & VECTOR STORE
# ============================================================

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma


class MovieDataLoader:
    """
    Memuat dan memproses dataset film dari CSV.
    Auto-normalisasi nama kolom dari berbagai format (IMDB/TMDB/Kaggle).
    Mendukung kolom dataset: names, date_x, score, genre, overview, crew,
    orig_title, status, orig_lang, budget_x, revenue, country.
    """

    # Mapping nama kolom alternatif → nama standar
    COLUMN_ALIASES = {
        'title':    ['names', 'title', 'movie_title', 'name', 'film_title', 'Series_Title'],
        'year':     ['date_x', 'year', 'release_year', 'release_date', 'Released_Year'],
        'genre':    ['genre', 'genres', 'Genre'],
        'rating':   ['score', 'rating', 'imdb_rating', 'vote_average', 'IMDB_Rating'],
        'overview': ['overview', 'description', 'plot', 'synopsis', 'Overview'],
        'crew':     ['crew', 'director', 'directors', 'Director'],   # ← kolom asli: crew
        'cast':     ['cast', 'actors', 'stars', 'Star1', 'Actors'],  # tidak ada di dataset → Unknown
        'orig_title': ['orig_title', 'original_title'],
        'country':  ['country', 'production_countries'],
        'language': ['orig_lang', 'original_language', 'language'],
        'status':   ['status', 'Status'],
        'budget':   ['budget_x', 'budget'],
        'revenue':  ['revenue', 'Revenue'],
    }

    def load(self, filepath: str) -> pd.DataFrame:
        """Load CSV dan normalisasi kolom."""
        df = pd.read_csv(filepath, encoding='utf-8', on_bad_lines='skip')
        return self._normalize(df)

    def _normalize(self, df: pd.DataFrame) -> pd.DataFrame:
        df.columns = [c.strip() for c in df.columns]
        col_lower = {c.lower(): c for c in df.columns}

        rename_map = {}
        for target, candidates in self.COLUMN_ALIASES.items():
            for cand in candidates:
                if cand in df.columns:
                    rename_map[cand] = target
                    break
                elif cand.lower() in col_lower:
                    rename_map[col_lower[cand.lower()]] = target
                    break

        df = df.rename(columns=rename_map)

        # Kolom wajib — sesuai kolom dataset yang tersedia
        required = ['title', 'year', 'genre', 'rating', 'overview', 'crew']
        optional = ['cast', 'orig_title', 'country', 'language', 'status', 'budget', 'revenue']

        for col in required:
            if col not in df.columns:
                df[col] = 'Unknown'
            else:
                df[col] = df[col].fillna('Unknown').astype(str).str.strip()

        for col in optional:
            if col not in df.columns:
                df[col] = 'Unknown'
            else:
                df[col] = df[col].fillna('Unknown').astype(str).str.strip()

        df['year'] = pd.to_numeric(
            df['year'].astype(str).str.extract(r'(\d{4})')[0], errors='coerce'
        ).fillna(0).astype(int)

        df['rating'] = pd.to_numeric(df['rating'], errors='coerce').fillna(0.0).round(1)

        df = df.drop_duplicates(subset=['title', 'year'])
        df = df[df['title'] != 'Unknown']

        print(f'📊 Dataset: {len(df)} film | Tahun: {df[df["year"]>0]["year"].min()}–{df["year"].max()}')
        print(f'   Kolom tersedia: {list(df.columns)}')
        return df

    def to_documents(self, df: pd.DataFrame) -> List[Document]:
        documents = []
        for _, row in df.iterrows():
            content = (
                f"Title: {row['title']}\n"
                f"Original Title: {row.get('orig_title', 'Unknown')}\n"
                f"Year: {row['year']}\n"
                f"Genre: {row['genre']}\n"
                f"Crew: {row['crew']}\n"          # ← pakai 'crew', bukan 'director'
                f"Rating: {row['rating']}/10\n"
                f"Overview: {row['overview']}\n"
                f"Country: {row.get('country', 'Unknown')}\n"
                f"Language: {row.get('language', 'Unknown')}\n"
                f"Status: {row.get('status', 'Unknown')}"
            )
            metadata = {
                'title':      str(row['title']),
                'orig_title': str(row.get('orig_title', 'Unknown')),
                'year':       int(row['year']),
                'genre':      str(row['genre']).lower(),
                'rating':     float(row['rating']),
                'crew':       str(row['crew']),         # ← crew sebagai metadata
                'director':   str(row['crew']),         # ← alias agar filter 'director' tetap jalan
                'cast':       str(row['crew']).lower(), # ← fallback cast ke crew
                'country':    str(row.get('country', 'Unknown')).lower(),
                'language':   str(row.get('language', 'Unknown')).lower(),
                'status':     str(row.get('status', 'Unknown')).lower(),
            }
            documents.append(Document(page_content=content, metadata=metadata))
        return documents


class VectorStoreManager:
    """Mengelola ChromaDB vector store dengan embedding lokal (HuggingFace)."""

    def __init__(self, persist_dir: str = CHROMA_PERSIST_DIR):
        self.persist_dir = persist_dir
        print('⏳ Memuat embedding model (download sekali ~30MB)...')
        self.embeddings = HuggingFaceEmbeddings(
            model_name=EMBEDDING_MODEL,
            model_kwargs={'device': 'cpu'},
            encode_kwargs={'normalize_embeddings': True}
        )
        print(f'✅ Embedding model siap: {EMBEDDING_MODEL}')
        self.vectorstore: Optional[Chroma] = None

    def build(self, documents: List[Document], batch_size: int = 100) -> None:
        """Bangun vector store baru dari documents (embed dalam batch)."""
        print(f'⏳ Membangun vector store dari {len(documents)} dokumen...')
        self.vectorstore = Chroma.from_documents(
            documents=documents[:batch_size],
            embedding=self.embeddings,
            persist_directory=self.persist_dir,
            collection_name='movies'
        )
        for i in range(batch_size, len(documents), batch_size):
            batch = documents[i:i + batch_size]
            self.vectorstore.add_documents(batch)
            print(f'   Batch {i//batch_size + 1}: {min(i + batch_size, len(documents))}/{len(documents)}')
        print(f'✅ Vector store selesai dibangun di: {self.persist_dir}')

    def load(self) -> None:
        """Load vector store yang sudah ada."""
        self.vectorstore = Chroma(
            persist_directory=self.persist_dir,
            embedding_function=self.embeddings,
            collection_name='movies'
        )
        count = self.vectorstore._collection.count()
        print(f'✅ Vector store dimuat: {count} dokumen dari {self.persist_dir}')

    def build_or_load(self, documents: List[Document]) -> None:
        """Otomatis: load jika vector store valid, build jika belum."""
        chroma_db_file = os.path.join(self.persist_dir, 'chroma.sqlite3')

        if os.path.exists(self.persist_dir) and os.path.exists(chroma_db_file):
            self.load()
            if self.vectorstore._collection.count() > 0:
                print(f'💡 Tip: set FORCE_REBUILD=True untuk rebuild ulang')
                return
            print('⚠️  Vector store kosong, rebuilding...')

        self.build(documents)


print('✅ Cell 3: MovieDataLoader & VectorStoreManager siap')


✅ Cell 3: MovieDataLoader & VectorStoreManager siap


## 🧠 Cell 4: MODULE 1 — Memory & Guardrails (BARU)

> **`ChatMemory`**: Menyimpan riwayat percakapan dengan sliding window.
> Setiap query dikirim bersama history sehingga chatbot memahami konteks
> ('film tadi', 'yang kedua', dll).
>
> **`TopicGuardrail`**: Layer pertahanan berlapis:
> 1. **Fast-path keyword** — langsung allow/reject tanpa LLM call
> 2. **LLM classifier** — untuk kasus ambigu dengan context history
> 3. **Fail-open** — jika LLM error, query tetap diproses (UX > keamanan ketat)

In [16]:
# ============================================================
# CELL 4 — MODULE 1: MEMORY & GUARDRAILS
# ============================================================


class ChatMemory:
    """
    Manajemen riwayat percakapan multi-turn.
    Menyimpan N turn terakhir sebagai sliding window.
    """

    def __init__(self, max_turns: int = MEMORY_WINDOW_TURNS):
        self.history: List[Dict[str, str]] = []
        self.max_turns = max_turns
        # Simpan hasil pencarian terakhir untuk referensi multi-turn
        self.last_docs: List[Document] = []
        self.last_titles: List[str] = []   # untuk deteksi "film no 3" dll

    def save_results(self, docs: List[Document]) -> None:
        """Simpan hasil search terakhir."""
        self.last_docs = docs
        self.last_titles = [d.metadata.get('title', '') for d in docs]

    def get_last_docs(self) -> List[Document]:
        return self.last_docs

    def get_last_titles_context(self) -> str:
        """Format untuk dimasukkan ke prompt router."""
        if not self.last_titles:
            return ''
        lines = [f'{i+1}. {t}' for i, t in enumerate(self.last_titles)]
        return 'Film dari hasil pencarian sebelumnya:\n' + '\n'.join(lines)

    # ── Penambahan pesan ──────────────────────────────────────
    def add_user(self, message: str) -> None:
        self.history.append({'role': 'user', 'content': message})
        self._trim()

    def add_assistant(self, message: str) -> None:
        self.history.append({'role': 'assistant', 'content': message})

    def _trim(self) -> None:
        """Pertahankan hanya N turn terakhir."""
        max_messages = self.max_turns * 2
        if len(self.history) > max_messages:
            self.history = self.history[-max_messages:]

    # ── Akses history ─────────────────────────────────────────
    def get_history(self) -> List[Dict[str, str]]:
        return self.history.copy()

    def get_recent_context(self, n_turns: int = 3) -> str:
        """Ambil N turn terakhir sebagai string (untuk prompt context)."""
        recent = self.history[-(n_turns * 2):]
        if not recent:
            return '(Tidak ada riwayat percakapan)'
        lines = []
        for msg in recent:
            role = 'User' if msg['role'] == 'user' else 'Bot'
            lines.append(f"{role}: {msg['content'][:250]}")
        return '\n'.join(lines)

    def clear(self) -> None:
        self.history = []

    def __len__(self) -> int:
        return len(self.history) // 2  # Jumlah turn


class TopicGuardrail:
    """
    Guardrail berlapis untuk memastikan chatbot hanya menjawab topik film.

    Lapisan 1: Keyword whitelist  → langsung ALLOW
    Lapisan 2: Keyword blacklist  → langsung REJECT
    Lapisan 3: LLM classifier    → untuk kasus ambigu
    Lapisan 4: Fail-open fallback → jika LLM error, izinkan query
    """

    # Kata kunci yang PASTI terkait film
    MOVIE_WHITELIST = {
        # Bahasa Indonesia
        'film', 'movie', 'sinema', 'bioskop', 'nonton', 'tonton',
        'rekomendasi', 'rekomendasikan', 'saran',
        'aktor', 'aktris', 'bintang', 'sutradara', 'pemain',
        'genre', 'alur', 'cerita', 'plot', 'ending', 'adegan',
        'horror', 'horor', 'action', 'thriller', 'comedy', 'komedi',
        'drama', 'romantis', 'animasi', 'dokumenter', 'sci-fi',
        'marvel', 'dc comics', 'disney', 'pixar', 'studio ghibli',
        'oscar', 'golden globe', 'imdb', 'rotten tomatoes',
        'sequel', 'prequel', 'remake', 'spin-off', 'franchise',
        'box office', 'trailer', 'cameo', 'karakter',
        # Bahasa Inggris
        'actor', 'actress', 'director', 'cinema', 'watch',
        'series', 'movie series', 'tv show', 'streaming',
        'screenplay', 'cinematography', 'soundtrack',
    }

    # Kata kunci yang PASTI di luar topik film
    OFF_TOPIC_BLACKLIST = {
        # Kuliner
        'resep', 'masak', 'makanan', 'kuliner', 'restoran', 'catering',
        # Politik
        'politik', 'pemilu', 'pilpres', 'presiden', 'legislatif', 'korupsi',
        # Teknologi/Coding
        'coding', 'programming', 'source code', 'javascript', 'python',
        'database', 'sql', 'machine learning', 'neural network',
        # Kecantikan
        'skincare', 'makeup', 'kosmetik', 'kecantikan', 'serum',
        # Keuangan
        'saham', 'investasi', 'kripto', 'crypto', 'bitcoin', 'forex',
        'deposito', 'tabungan', 'pinjaman',
        # Olahraga (selain yang ada filmnya)
        'liga sepakbola', 'pertandingan bola', 'skor bola',
        # Kesehatan
        'obat', 'penyakit', 'dokter', 'resep dokter', 'gejala',
    }

    def __init__(self, llm_client):
        self.client = llm_client

    def check(self, query: str, history_context: str) -> Tuple[bool, str]:
        """
        Returns:
            (True, '')           → query relevan, lanjutkan
            (False, pesan_tolak) → query ditolak, tampilkan pesan
        """
        q = query.lower()

        # ── Layer 1: Whitelist cepat ───────────────────────────
        if any(kw in q for kw in self.MOVIE_WHITELIST):
            return True, ''

        # ── Layer 2: Blacklist cepat ───────────────────────────
        if any(kw in q for kw in self.OFF_TOPIC_BLACKLIST):
            return False, self._get_rejection_message()

        # ── Layer 3: LLM classifier (kasus ambigu) ─────────────
        return self._llm_classify(query, history_context)

    def _llm_classify(self, query: str, history: str) -> Tuple[bool, str]:
        """Gunakan LLM untuk klasifikasi kontekstual."""
        prompt = f"""Kamu adalah classifier topik untuk chatbot film.

Riwayat percakapan terakhir:
{history}

Query baru: "{query}"

Tugasmu: Apakah query ini relevan dengan topik FILM atau SINEMA?

PENTING: Pertimbangkan konteks!
- 'yang mana lebih bagus?' → RELEVAN jika konteks sebelumnya tentang film
- 'siapa pemerannya?' → RELEVAN (aktor film)
- 'berapa harganya?' → TIDAK relevan (harga tiket mungkin oke, tapi cek konteks)

Jawab hanya JSON:
{{"is_movie_related": true/false, "reason": "alasan singkat dalam 1 kalimat"}}"""

        try:
            response = self.client.chat.completions.create(
                model=LLM_MODEL,
                messages=[{'role': 'user', 'content': prompt}],
                max_tokens=80,
                temperature=0,
                response_format={'type': 'json_object'}
            )
            result = json.loads(response.choices[0].message.content)
            is_relevant = result.get('is_movie_related', True)
            print(f'   🔍 Guardrail LLM: {"ALLOW" if is_relevant else "REJECT"} — {result.get("reason", "")}')
            return (True, '') if is_relevant else (False, self._get_rejection_message())
        except Exception as e:
            # ── Layer 4: Fail-open ─────────────────────────────
            print(f'   ⚠️ Guardrail LLM error (fail-open): {e}')
            return True, ''

    @staticmethod
    def _get_rejection_message() -> str:
        return (
            '🎬 Maaf, saya hanya bisa membantu tentang **film dan sinema**.\n\n'
            'Sepertinya pertanyaan Anda di luar topik tersebut. '
            'Saya siap membantu dengan:\n\n'
            '• 🎭 **Rekomendasi film** — berdasarkan genre, mood, atau vibe\n'
            '• 🔍 **Info film** — plot, cast, rating, sutradara\n'
            '• ⭐ **Perbandingan** — mana film yang lebih bagus?\n'
            '• 🗓️ **Film by era** — film terbaik tahun tertentu\n\n'
            'Ada film apa yang ingin Anda tanyakan? 😊'
        )


print('✅ Cell 4: ChatMemory & TopicGuardrail siap')

✅ Cell 4: ChatMemory & TopicGuardrail siap


## 🧭 Cell 5: MODULE 2 — Query Router (BARU)

> **`QueryIntent`**: Dataclass yang merepresentasikan hasil analisa query.
>
> **`QueryRouter`**: Menggunakan LLM untuk mengekstrak intent terstruktur:
> - **`metadata_filter`**: Cari berdasarkan kriteria spesifik (genre, rating, tahun, aktor)
> - **`semantic_search`**: Cari berdasarkan kemiripan plot/vibe
> - **`hybrid`**: Kombinasi keduanya
> - **`needs_external`**: Flag untuk trigger fallback API (film > 2023)

In [17]:
@dataclass
class QueryIntent:
    """Hasil analisa query — menentukan strategi pencarian."""
    strategy: str            # 'metadata_filter' | 'semantic_search' | 'hybrid'
    filters: Dict[str, Any] = field(default_factory=dict)
    semantic_query: str = ''
    limit: int = DEFAULT_RESULT_LIMIT
    needs_external: bool = False
    raw_query: str = ''
    explicit_title: str = ''

    def __post_init__(self):
        self.limit = max(1, min(20, int(self.limit) if self.limit is not None else DEFAULT_RESULT_LIMIT))

    def summary(self) -> str:
        parts = [f'strategy={self.strategy}', f'limit={self.limit}']
        if self.filters:
            parts.append(f'filters={self.filters}')
        if self.needs_external:
            parts.append('needs_external=True')
        if self.explicit_title:
            parts.append(f'explicit_title="{self.explicit_title}"')
        return ' | '.join(parts)


class QueryRouter:
    """
    Menganalisa query user untuk menentukan strategi pencarian optimal.

    Input : query string + riwayat percakapan
    Output: QueryIntent + mode keputusan via resolve_query_mode()
    """

    ROUTING_PROMPT = """\
Kamu adalah query analyzer untuk sistem pencarian film.

Riwayat percakapan:
{history}

Query user: "{query}"

Analisa query dan hasilkan JSON dengan struktur berikut:

{{
  "strategy": "metadata_filter" | "semantic_search" | "hybrid",
  "filters": {{
    "genre": null atau string,
    "year_from": null atau integer,
    "year_to": null atau integer,
    "min_rating": null atau float,
    "actor": null atau string,
    "director": null atau string
  }},
  "semantic_query": "kalimat deskriptif panjang untuk similarity search",
  "limit": integer,
  "needs_external": boolean,
  "explicit_title": null atau string
}}

ATURAN STRATEGI:
- 'metadata_filter': query spesifik berdasarkan atribut → "film comedy rating di atas 8"
- 'semantic_search': berbasis vibe/plot → "film seperti Interstellar", "film tentang balas dendam"
- 'hybrid': ada filter DAN semantic → "film sci-fi yang mirip Interstellar"

ATURAN explicit_title (PENTING - baca dengan teliti):
- Isi jika user menyebut judul film secara spesifik
- Contoh: "crew dari film Bloody Hell" → explicit_title: "Bloody Hell"
- Contoh: "berapa budget Memory?" → explicit_title: "Memory"
- Contoh: "film nomor 3" → explicit_title: null (referensi ke list, bukan judul)
- TANDA KUTIP: Teks dalam tanda kutip tunggal/ganda ADALAH judul film!
  - "film 'agak laen' membahas apa" → explicit_title: "agak laen"
  - 'film "comic 8" itu tentang apa' → explicit_title: "comic 8"
- JUDUL INDONESIA: Ekstrak judul film Indonesia meski terdengar umum:
  - "film agak laen", "film comic 8", "film qorin 2", "film satans slaves"
  - → explicit_title wajib diisi
- QUERY SIMILARITY: "film mirip X", "film seperti X", "ada film yang serupa dengan X"
  - → explicit_title: "X" (judul film rujukan), strategy: "semantic_search"

ATURAN semantic_query (PENTING):
- Harus berupa deskripsi PANJANG dan KAYA konten dalam BAHASA INGGRIS
- Contoh BURUK: "film mirip Interstellar"
- Contoh BAGUS: "science fiction space exploration time relativity emotional journey stunning visuals"
- Untuk similarity query: deskripsikan PLOT/VIBE dari judul rujukan, BUKAN judulnya

ATURAN needs_external=true:
- Menyebut tahun 2024, 2025, 2026
- User bertanya 'film terbaru' tanpa batas tahun jelas

ATURAN limit:
- 'top 10', 'sepuluh film' → limit: 10
- 'beberapa' → limit: 5
- tidak disebutkan → limit: {default_limit}

Jawab HANYA dengan JSON valid tanpa penjelasan tambahan."""

    # Pattern untuk referensi NOMOR/INDEX dari list sebelumnya
    INDEX_REF_PATTERNS = [
        r'\bno\.?\s*\d+\b',
        r'\bnomor\s*\d+\b',
        r'\bfilm\s+(?:ke-?\d+|pertama|kedua|ketiga|keempat|kelima)\b',
        r'\byang\s+(?:pertama|kedua|ketiga|keempat|kelima)\b',
        r'\bdi\s+(?:list|urutan|daftar)\s+(?:nomor\s*)?\d+\b',
    ]

    # Pattern untuk pertanyaan lanjutan GENERIK
    GENERIC_FOLLOWUP_PATTERNS = [
        r'\bjelaskan\s+(?:lebih|detail)\b',
        r'\bceritakan\s+(?:tentang|lebih|apa)\b',
        r'\bbercerita\s+tentang\b',
        r'\binfo\s+(?:lebih|detail|lengkap)\b',
        r'\btayang\s+(?:kapan|tahun)\b',
        r'\btahun\s+berapa\b',
        r'\bkapan\s+tayang\b',
        r'\bberapa\s+(?:rating|tahun|durasi)\b',
        r'\bsiapa\s+(?:saja\s+)?(?:crew|sutradara|pemain|cast|aktor)\b',
        r'\bberapa\s+(?:revenue|budget|pendapatan)\b',
        r'\brevenue\b',
        r'\bbudget\b',
        r'\bpendapatan\b',
        r'\bcontext\s+(?:apa|yang)\b',
        r'\blist(?:nya)?\s+(?:apa|tadi)\b',
        r'\bapa\s+saja\s+(?:yang|film|tadi)\b',
        r'\btadi\s+(?:kan|kamu|ada)\b',
        r'\byang\s+(?:tadi|sebelumnya|kamu\s+(?:sebut|rekomen))\b',
        r'\bdalam\s+list\b',
        r'\bdari\s+(?:list|daftar)\s+(?:tadi|sebelumnya)\b',
    ]

    # Pattern similarity (rekomendasi film mirip) — diperluas
    SIMILARITY_PATTERNS = [
        r'\bmirip\s+(?:dengan\s+)?\w',
        r'\bseperti\b',
        r'\bsimilar\s+to\b',
        r'\brekomendasi.{0,20}mirip\b',
        r'\bfilm\s+lain\s+yang\b',
        r'\byang\s+serupa\b',
        r'\bsejenisnya\b',
        r'\bsama\s+vibe\b',
        r'\bsama\s+genre\b',
        r'\bada\s+(?:film|movie)\s+(?:lain|lainnya)\b',
    ]

    # Stop words untuk title verification (jangan dipakai untuk match)
    TITLE_STOP_WORDS = {
        'the', 'a', 'an', 'in', 'of', 'and', 'or', 'to', 'is', 'are',
        'was', 'be', 'as', 'at', 'by', 'for', 'on', 'with', 'from',
    }

    def __init__(self, llm_client):
        self.client = llm_client

    def route(self, query: str, history_context: str) -> QueryIntent:
        """Analisa query via LLM dan kembalikan QueryIntent."""
        prompt = self.ROUTING_PROMPT.format(
            history=history_context,
            query=query,
            default_limit=DEFAULT_RESULT_LIMIT
        )
        try:
            response = self.client.chat.completions.create(
                model=LLM_MODEL,
                messages=[{'role': 'user', 'content': prompt}],
                max_tokens=300,
                temperature=0,
                response_format={'type': 'json_object'}
            )
            raw = response.choices[0].message.content
            data = json.loads(raw)

            raw_limit = data.get('limit')
            safe_limit = int(raw_limit) if raw_limit is not None else DEFAULT_RESULT_LIMIT

            return QueryIntent(
                strategy=data.get('strategy', 'semantic_search'),
                filters=self._clean_filters(data.get('filters', {})),
                semantic_query=data.get('semantic_query', query),
                limit=safe_limit,
                needs_external=bool(data.get('needs_external', False)),
                raw_query=query,
                explicit_title=data.get('explicit_title') or '',
            )
        except Exception as e:
            print(f'   ⚠️ Router error, fallback semantic: {e}')
            needs_ext = bool(re.search(r'202[4-9]|203\d', query))
            return QueryIntent(
                strategy='semantic_search',
                semantic_query=query,
                needs_external=needs_ext,
                raw_query=query
            )

    @staticmethod
    def _clean_filters(filters: Dict) -> Dict:
        """Hapus filter dengan nilai None/null."""
        return {k: v for k, v in filters.items() if v is not None}

    # ── Deteksi helpers ───────────────────────────────────────

    def is_index_ref(self, query: str) -> bool:
        q = query.lower()
        return any(re.search(p, q) for p in self.INDEX_REF_PATTERNS)

    def is_generic_followup(self, query: str) -> bool:
        q = query.lower()
        return any(re.search(p, q) for p in self.GENERIC_FOLLOWUP_PATTERNS)

    def is_similarity_query(self, query: str) -> bool:
        q = query.lower()
        return any(re.search(p, q) for p in self.SIMILARITY_PATTERNS)

    def extract_referenced_index(self, query: str) -> Optional[int]:
        q = query.lower()
        m = re.search(r'(?:no\.?\s*|nomor\s*|ke-?)(\d+)', q)
        if m:
            return int(m.group(1)) - 1
        ordinals = {'pertama': 0, 'kedua': 1, 'ketiga': 2, 'keempat': 3, 'kelima': 4}
        for word, idx in ordinals.items():
            if word in q:
                return idx
        return None

    def find_title_in_cache(self, query: str, last_titles: List[str]) -> Optional[int]:
        if not last_titles:
            return None
        q = query.lower()
        for i, title in enumerate(last_titles):
            title_words = title.lower().split()
            if len(title_words) >= 2:
                for j in range(len(title_words) - 1):
                    bigram = f"{title_words[j]} {title_words[j+1]}"
                    if bigram in q:
                        return i
            elif len(title_words) == 1 and title_words[0] in q:
                return i
        return None

    def verify_title_match(self, explicit_title: str, found_title: str) -> bool:
        """
        [BUG 3 FIX] Verifikasi apakah film yang ditemukan COCOK dengan judul yang dicari.
        Gunakan word-level intersection, abaikan stop words dan kata pendek.
        """
        if not explicit_title or not found_title:
            return False
        exp_words = {
            w for w in explicit_title.lower().split()
            if len(w) > 2 and w not in self.TITLE_STOP_WORDS
        }
        if not exp_words:
            # Semua kata pendek (e.g. "It") — lakukan exact match
            return explicit_title.lower().strip() in found_title.lower()
        found_lower = found_title.lower()
        # Setidaknya SATU kata bermakna dari judul yang dicari harus ada di judul yang ditemukan
        return any(w in found_lower for w in exp_words)

    # ── [BUG 1 FIX] resolve_query_mode — urutan prioritas diperbaiki ─────────
    def resolve_query_mode(
        self,
        query: str,
        intent: QueryIntent,
        last_titles: List[str],
        last_docs_count: int,
    ) -> str:
        """
        Mengembalikan salah satu dari mode:
          'cache_index'  → user sebut nomor dari list rekomendasi sebelumnya
          'cache_title'  → user sebut judul yang ada di cache list
          'cache_follow' → lanjutan generik, pakai semua film dari cache
          'similarity'   → rekomendasi mirip, SELALU butuh semantic search baru
          'db_title'     → user sebut judul eksplisit, cari ke DB
          'db_search'    → cari ke DB dengan filter/semantic/hybrid
          'external'     → butuh API luar (film > 2023 atau tidak ada di DB)

        URUTAN PRIORITAS (diperbaiki dari versi sebelumnya):
          1. Index reference (nomor X)        — paling spesifik
          2. [BUG 1 FIX] Similarity query    — SEBELUM cache_title!
          3. Title match di cache             — judul dari list sebelumnya
          4. Explicit title → db_title        — judul baru tidak di cache
          5. Generic followup                 — pertanyaan lanjutan tanpa judul
          6. Needs external                   — film luar DB
          7. Default: db_search
        """
        has_cache = last_docs_count > 0

        # Priority 1: Index reference (nomor X) — paling spesifik, cek duluan
        if has_cache and self.is_index_ref(query):
            ref_idx = self.extract_referenced_index(query)
            if ref_idx is not None and ref_idx < last_docs_count:
                return 'cache_index'

        # [BUG 1 FIX] Priority 2: Similarity query → SELALU lakukan semantic search BARU
        # WAJIB di atas cache_title agar "film mirip X" tidak salah masuk cache_title
        # meskipun X ada di cache. Similarity HARUS cari film LAIN, bukan X itu sendiri.
        if self.is_similarity_query(query):
            return 'similarity'

        # Priority 3: Title match di cache list
        if has_cache:
            lookup_key = intent.explicit_title if intent.explicit_title else query
            title_match = self.find_title_in_cache(lookup_key, last_titles)
            if title_match is not None:
                return 'cache_title'

        # Priority 4: Explicit title tidak ada di cache → cari DB
        if intent.explicit_title:
            return 'db_title'

        # Priority 5: Generic followup (tanpa judul eksplisit baru)
        if has_cache and self.is_generic_followup(query) and not intent.explicit_title:
            return 'cache_follow'

        # Priority 6: External API
        if intent.needs_external:
            return 'external'

        # Default: search ke DB
        return 'db_search'


print('✅ Cell 5: QueryIntent & QueryRouter siap (v3 — BUG 1,3,4 fixed)')


✅ Cell 5: QueryIntent & QueryRouter siap (v3 — BUG 1,3,4 fixed)


## 🔍 Cell 6: MODULE 2 — Search Engine (DIPERBAIKI)

> **Perubahan dari versi lama:**
> - Ditambahkan 3 mode pencarian: `_metadata_search`, `_semantic_search`, `_hybrid_search`
> - **Dynamic output limit**: Jumlah hasil menyesuaikan `intent.limit`
> - **Re-ranking by rating**: Dari semantic candidates, pilih top-N berdasarkan rating tertinggi
> - **ChromaDB filter builder**: Konversi filters dict → ChromaDB `where` clause

In [18]:
# ============================================================
# CELL 6 — MODULE 2: SEARCH ENGINE
# ============================================================


class MovieSearchEngine:
    """
    Mesin pencarian film dengan 3 strategi:
    1. metadata_filter  — filter ChromaDB berdasarkan atribut
    2. semantic_search  — similarity search + re-rank by rating
    3. hybrid           — semantic search + post-filter metadata
    """

    def __init__(self, vsm: VectorStoreManager):
        self.vsm = vsm

    def search(self, intent: QueryIntent) -> List[Document]:
        """Entry point: pilih strategi berdasarkan intent."""
        if intent.strategy == 'metadata_filter':
            return self._metadata_search(intent)
        elif intent.strategy == 'semantic_search':
            return self._semantic_search(intent)
        else:  # hybrid
            return self._hybrid_search(intent)

    # ── Strategi 1: Metadata Filter ───────────────────────────
    def _metadata_search(self, intent: QueryIntent) -> List[Document]:
        candidate_limit = max(intent.limit * 5, 50)

        try:
            # Selalu mulai dari semantic search dulu untuk dapat kandidat
            query = intent.semantic_query or intent.raw_query or 'popular movies'
            candidates = self.vsm.vectorstore.similarity_search(
                query, k=candidate_limit
            )

            # Post-filter metadata (sama seperti hybrid)
            if intent.filters:
                filtered = [
                    doc for doc in candidates
                    if self._passes_filter(doc.metadata, intent.filters)
                ]
                # Jika hasil terlalu sedikit, ambil semua kandidat tanpa filter
                if len(filtered) < intent.limit:
                    print(f'   ⚠️ Filter ketat ({len(filtered)} hasil), melonggarkan...')
                    # Coba ambil lebih banyak kandidat
                    candidates = self.vsm.vectorstore.similarity_search(
                        query, k=candidate_limit * 3
                    )
                    filtered = [
                        doc for doc in candidates
                        if self._passes_filter(doc.metadata, intent.filters)
                    ]
            else:
                filtered = candidates

            filtered.sort(key=lambda d: d.metadata.get('rating', 0), reverse=True)
            return filtered[:intent.limit]

        except Exception as e:
            print(f'   ⚠️ Metadata search error: {e}')
            return self._semantic_search(intent)

    # ── Strategi 2: Semantic Search ───────────────────────────
    def _semantic_search(self, intent: QueryIntent) -> List[Document]:
        """
        Semantic similarity search.
        Ambil top-MAX_SEMANTIC_CANDIDATES, lalu re-rank by rating.
        """
        query = intent.semantic_query or intent.raw_query
        try:
            candidates = self.vsm.vectorstore.similarity_search(
                query,
                k=MAX_SEMANTIC_CANDIDATES
            )
            # Re-rank: dari hasil semantik, pilih yang rating tertinggi
            candidates.sort(key=lambda d: d.metadata.get('rating', 0), reverse=True)
            return candidates[:intent.limit]
        except Exception as e:
            print(f'   ⚠️ Semantic search error: {e}')
            return []

    # ── Strategi 3: Hybrid ────────────────────────────────────
    def _hybrid_search(self, intent: QueryIntent) -> List[Document]:
        """
        Gabungan: semantic search + post-filter berdasarkan metadata.
        Lebih fleksibel dari metadata_filter karena dimulai dari embedding.
        """
        query = intent.semantic_query or intent.raw_query
        try:
            # Step 1: Dapatkan kandidat semantik
            candidates = self.vsm.vectorstore.similarity_search(
                query,
                k=MAX_SEMANTIC_CANDIDATES * 2
            )
            # Step 2: Post-filter berdasarkan metadata
            filtered = []
            for doc in candidates:
                if self._passes_filter(doc.metadata, intent.filters):
                    filtered.append(doc)

            # Step 3: Sort by rating
            filtered.sort(key=lambda d: d.metadata.get('rating', 0), reverse=True)

            # Step 4: Jika hasil terlalu sedikit, longgarkan ke semantic biasa
            if len(filtered) < intent.limit // 2:
                print('   ⚠️ Hybrid: hasil terlalu sedikit, fallback ke semantic')
                return self._semantic_search(intent)

            return filtered[:intent.limit]
        except Exception as e:
            print(f'   ⚠️ Hybrid search error: {e}')
            return self._semantic_search(intent)

    # ── Helper: Build ChromaDB where clause ───────────────────
    @staticmethod
    def _build_where_clause(filters: Dict) -> Optional[Dict]:
        """Konversi dict filters → ChromaDB $and/$or where clause."""
        conditions = []

        if filters.get('genre'):
            conditions.append({'genre': {'$contains': filters['genre'].lower()}})

        if filters.get('min_rating') is not None:
            conditions.append({'rating': {'$gte': float(filters['min_rating'])}})

        if filters.get('year_from') is not None:
            conditions.append({'year': {'$gte': int(filters['year_from'])}})

        if filters.get('year_to') is not None:
            conditions.append({'year': {'$lte': int(filters['year_to'])}})

        if filters.get('actor'):
            conditions.append({'cast': {'$contains': filters['actor'].lower()}})

        if filters.get('director'):
            conditions.append({'director': {'$contains': filters['director'].lower()}})

        if not conditions:
            return None
        if len(conditions) == 1:
            return conditions[0]
        return {'$and': conditions}

    # ── Helper: Post-filter checker ───────────────────────────
    @staticmethod
    def _passes_filter(metadata: Dict, filters: Dict) -> bool:
        """Filter dengan dukungan multi-genre (OR logic antar genre)."""
        genre_meta = str(metadata.get('genre', '')).lower()

        if filters.get('genre'):
            # Pisahkan multi-genre dari filter: "comedy horror" → ['comedy', 'horror']
            requested_genres = [g.strip() for g in re.split(r'[\s,|/]+', filters['genre'].lower()) if g.strip()]
            # Semua genre yang diminta harus ada (AND logic)
            if not all(g in genre_meta for g in requested_genres):
                return False

        if filters.get('min_rating') is not None:
            if metadata.get('rating', 0) < filters['min_rating']:
                return False
        if filters.get('year_from') is not None:
            if metadata.get('year', 0) < filters['year_from']:
                return False
        if filters.get('year_to') is not None:
            if metadata.get('year', 9999) > filters['year_to']:
                return False
        if filters.get('actor'):
            if filters['actor'].lower() not in str(metadata.get('cast', '')).lower():
                return False
        if filters.get('director'):
            if filters['director'].lower() not in str(metadata.get('director', '')).lower():
                return False
        return True


print('✅ Cell 6: MovieSearchEngine siap (metadata | semantic | hybrid)')

✅ Cell 6: MovieSearchEngine siap (metadata | semantic | hybrid)


## 🌐 Cell 7: MODULE 3 — Fallback Mechanism (BARU)

> **Kapan fallback diaktifkan:**
> 1. `intent.needs_external = True` (film > 2023)
> 2. Hasil pencarian lokal = 0
> 3. Hasil lokal < setengah dari `intent.limit`
>
> **Sumber fallback** (berurutan):
> 1. **TMDB API** — database film terlengkap, gratis, mendukung Bahasa Indonesia
> 2. **OMDb API** — IMDB data, detail per film
>
> **Sitasi wajib**: Setiap jawaban dari sumber eksternal **harus** menyertakan
> `Sumber: [nama API]` di akhir response.

In [19]:
# ============================================================
# CELL 7 — MODULE 3: FALLBACK EXTERNAL API
# ============================================================


class ExternalMovieAPI:
    """
    Fallback ke TMDB / OMDb untuk film yang tidak ada di database lokal.
    Selalu menyertakan sitasi sumber di output.
    """

    TMDB_BASE = 'https://api.themoviedb.org/3'
    OMDB_BASE = 'http://www.omdbapi.com'
    TIMEOUT   = 10  # detik

    def __init__(self, tmdb_key: str = '', omdb_key: str = ''):
        self.tmdb_key = tmdb_key
        self.omdb_key = omdb_key

    def is_available(self) -> bool:
        return bool(self.tmdb_key or self.omdb_key)

    def search(self, query: str, intent: QueryIntent) -> Tuple[List[Dict], str]:
        """
        Cari film dari API eksternal.
        Returns: (list_film, nama_sumber)
        """
        # Prioritas 1: TMDB
        if self.tmdb_key:
            results = self._tmdb_search(query, intent)
            if results:
                return results, 'TMDB (The Movie Database)'

        # Prioritas 2: OMDb
        if self.omdb_key:
            results = self._omdb_search(query, intent)
            if results:
                return results, 'OMDb (Open Movie Database)'

        return [], 'Tidak ada sumber eksternal tersedia'

    # ── TMDB Search ───────────────────────────────────────────
    def _tmdb_search(self, query: str, intent: QueryIntent) -> List[Dict]:
        try:
            params = {
                'api_key': self.tmdb_key,
                'query': query,
                'language': 'id-ID',   # Bahasa Indonesia
                'page': 1,
                'include_adult': 'false'
            }

            # Tambahkan filter tahun jika ada
            if intent.filters.get('year_from'):
                params['primary_release_date.gte'] = f"{intent.filters['year_from']}-01-01"
            if intent.filters.get('year_to'):
                params['primary_release_date.lte'] = f"{intent.filters['year_to']}-12-31"

            resp = requests.get(
                f'{self.TMDB_BASE}/search/movie',
                params=params,
                timeout=self.TIMEOUT
            )
            resp.raise_for_status()
            data = resp.json()

            movies = []
            for m in data.get('results', [])[:intent.limit]:
                year = m.get('release_date', '')[:4] or 'Unknown'
                movies.append({
                    'title':    m.get('title', 'Unknown'),
                    'year':     year,
                    'rating':   round(m.get('vote_average', 0), 1),
                    'overview': m.get('overview', 'Deskripsi tidak tersedia')[:300],
                    'genre':    'Lihat TMDB',
                    'source':   'TMDB'
                })
            return movies

        except requests.Timeout:
            print('   ⚠️ TMDB timeout')
        except requests.HTTPError as e:
            print(f'   ⚠️ TMDB HTTP error: {e}')
        except Exception as e:
            print(f'   ⚠️ TMDB error: {e}')
        return []

    # ── OMDb Search ───────────────────────────────────────────
    def _omdb_search(self, query: str, intent: 'QueryIntent') -> List[Dict]:
        try:
            resp = requests.get(
                self.OMDB_BASE,
                params={'apikey': self.omdb_key, 's': query, 'type': 'movie'},
                timeout=self.TIMEOUT
            )
            resp.raise_for_status()
            data = resp.json()

            if data.get('Response') == 'False':
                return []

            movies = []
            for item in data.get('Search', [])[:intent.limit]:
                imdb_id = item.get('imdbID', '')   # ← ambil imdbID dulu
                detail = self._omdb_detail(imdb_id) # ← baru pass ke detail
                if detail:
                    movies.append(detail)
            return movies

        except Exception as e:
            print(f'   ⚠️ OMDb error: {e}')
        return []

    def _omdb_detail(self, imdb_id: str) -> Optional[Dict]:
        """Ambil detail film dari OMDb by IMDb ID."""
        if not imdb_id:
            return None
        try:
            resp = requests.get(
                self.OMDB_BASE,                              # ← URL bersih
                params={
                    'apikey': self.omdb_key,                 # ← key terpisah
                    'i': imdb_id,
                    'plot': 'short'
                },
                timeout=5
            )
            d = resp.json()
            if d.get('Response') == 'False':
                return None
            return {
                'title':    d.get('Title', 'Unknown'),
                'year':     d.get('Year', 'Unknown'),
                'rating':   d.get('imdbRating', 'N/A'),
                'genre':    d.get('Genre', 'Unknown'),
                'overview': d.get('Plot', 'No description'),
                'director': d.get('Director', 'Unknown'),
                'cast':     d.get('Actors', 'Unknown'),
                'source':   'OMDb'
            }
        except Exception as e:
            print(f'   ⚠️ OMDb detail error: {e}')
            return None

    # ── Format untuk LLM context ──────────────────────────────
    @staticmethod
    def format_results(movies: List[Dict]) -> str:
        """Format hasil eksternal menjadi string konteks untuk LLM."""
        if not movies:
            return ''
        lines = []
        for i, m in enumerate(movies, 1):
            lines.append(f"{i}. Title: {m.get('title', 'Unknown')}")
            lines.append(f"   Year: {m.get('year', 'Unknown')}")
            lines.append(f"   Rating: {m.get('rating', 'N/A')} | Genre: {m.get('genre', 'N/A')}")
            overview = str(m.get('overview', ''))[:250]
            lines.append(f"   Overview: {overview}")
            if m.get('director') and m['director'] not in ('Unknown', 'Lihat TMDB'):
                lines.append(f"   Director: {m['director']}")
            lines.append('')
        return '\n'.join(lines)


print('✅ Cell 7: ExternalMovieAPI siap (TMDB + OMDb + sitasi)')

✅ Cell 7: ExternalMovieAPI siap (TMDB + OMDb + sitasi)


## 💬 Cell 8: Response Generator (DIPERBAIKI)

> **Perubahan dari versi lama:**
> - Sistem prompt di-refactor dengan instruksi anti-halusinasi eksplisit
> - Context injection terstruktur: local DB vs external API dipisahkan
> - Dynamic limit injected ke prompt untuk konsistensi output
> - Wajib include sitasi jika menggunakan data eksternal

In [20]:
class ResponseGenerator:
    """Menghasilkan jawaban natural dari context film menggunakan LLM."""

    SYSTEM_PROMPT = """\
Kamu adalah CineBot, asisten AI yang ramah dan berpengetahuan luas tentang film.

KEPRIBADIAN:
- Antusias saat membahas film, gunakan emoji film sesekali (🎬 🎭 ⭐ 🍿)
- Berbicara dalam Bahasa Indonesia yang natural
- Berikan jawaban yang informatif namun ringkas

ATURAN KETAT (ANTI-HALUSINASI):
1. HANYA gunakan informasi dari [CONTEXT] yang diberikan
2. JANGAN mengarang detail film (tanggal, cast, plot) yang tidak ada di context
3. Jika [CONTEXT] kosong atau bertuliskan "TIDAK ADA DATA": sampaikan film tidak ditemukan
4. JANGAN menyebutkan bahwa kamu 'mencari' atau 'menggunakan database'

[BUG 2 FIX] ATURAN URUTAN TAMPILAN (WAJIB DIIKUTI):
- Tampilkan film PERSIS dalam urutan yang sama dengan urutan di [CONTEXT]
- Film pertama di [CONTEXT] = Nomor 1, film kedua = Nomor 2, dan seterusnya
- DILARANG KERAS mengubah, membalik, atau menyusun ulang urutan film
- Urutan ini SANGAT PENTING karena user akan merujuk "nomor X" sesuai urutan tampilan

ATURAN JAWABAN BERDASARKAN JUMLAH DATA:
- Jika [CONTEXT] hanya berisi 1 film: jawab pertanyaan user khusus tentang film itu
- Jika [CONTEXT] berisi banyak film: tampilkan sesuai permintaan user
- Jika [CONTEXT] kosong: sampaikan film tidak ditemukan, JANGAN tawarkan film lain

ATURAN SITASI (WAJIB):
- Data dari Database Lokal → cantumkan di akhir: "\\n\\n---\\n📡 Sumber: Database Lokal"
- Data dari API eksternal → cantumkan: "\\n\\n---\\n📡 Sumber: [nama API]"

FORMAT REKOMENDASI:
🎬 **[Judul]** ([Tahun]) — ⭐ [Rating]/10
[Penjelasan singkat kenapa film ini cocok dengan permintaan user]"""

    def __init__(self, llm_client):
        self.client = llm_client

    def generate(
        self,
        query: str,
        local_docs: List[Document],
        external_data: Optional[Tuple[List[Dict], str]],
        history: List[Dict],
        intent: QueryIntent,
        query_mode: str = 'db_search',
    ) -> str:
        """Generate response dari context gabungan."""

        # ── Build context ──────────────────────────────────────
        context_sections = []
        has_external = False
        external_source = ''
        is_local = False

        if local_docs:
            local_text = '\n---\n'.join(doc.page_content for doc in local_docs)
            context_sections.append(
                f'=== DATA DARI DATABASE LOKAL ({len(local_docs)} film) ===\n{local_text}'
            )
            is_local = True

        if external_data and external_data[0]:
            ext_movies, ext_source = external_data
            ext_text = ExternalMovieAPI.format_results(ext_movies)
            context_sections.append(
                f'=== DATA DARI SUMBER EKSTERNAL: {ext_source} ===\n{ext_text}'
            )
            has_external = True
            external_source = ext_source

        context = '\n\n'.join(context_sections) if context_sections else (
            'TIDAK ADA DATA FILM yang ditemukan untuk query ini.'
        )

        # ── Build messages ─────────────────────────────────────
        messages = [{'role': 'system', 'content': self.SYSTEM_PROMPT}]
        messages.extend(history[-6:])

        # Instruksi sitasi
        if has_external:
            citation_instruction = (
                f'\nPENTING: Data dari "{external_source}" — '
                f'cantumkan "📡 Sumber: {external_source}" di akhir jawaban.'
            )
        elif is_local:
            citation_instruction = '\nPENTING: Data dari Database Lokal — cantumkan "📡 Sumber: Database Lokal" di akhir jawaban.'
        else:
            citation_instruction = ''

        # Instruksi limit adaptif berdasarkan mode
        n_docs = len(local_docs)
        if query_mode in ('cache_index', 'cache_title', 'db_title'):
            limit_instruction = (
                '- Jawab pertanyaan user tentang film yang ada di [CONTEXT]\n'
                '- Fokus ke informasi yang relevan dengan pertanyaan\n'
                '- JANGAN katakan "data tidak mencukupi" — gunakan semua info yang tersedia'
            )
        elif query_mode == 'cache_follow':
            limit_instruction = (
                f'- Jawab pertanyaan user berdasarkan {n_docs} film di [CONTEXT]\n'
                '- Gunakan semua informasi yang relevan dengan pertanyaan user'
            )
        else:
            limit_instruction = (
                f'- Tampilkan maksimal {intent.limit} film\n'
                f'- [BUG 2 FIX] WAJIB tampilkan film dalam urutan PERSIS sama dengan urutan di [CONTEXT]\n'
                f'  Film ke-1 di context = Nomor 1, film ke-2 = Nomor 2, dst. JANGAN ubah urutan!\n'
                '- JANGAN isi slot yang kosong dengan film fiktif atau tidak relevan'
            )

        user_content = (
            f'[CONTEXT]\n{context}\n[/CONTEXT]\n\n'
            f'Pertanyaan user: {query}\n\n'
            f'Instruksi:\n'
            f'{limit_instruction}'
            f'{citation_instruction}'
        )
        messages.append({'role': 'user', 'content': user_content})

        # ── Call LLM ───────────────────────────────────────────
        try:
            response = self.client.chat.completions.create(
                model=LLM_MODEL,
                messages=messages,
                max_tokens=1800,
                temperature=0.7
            )
            return response.choices[0].message.content
        except Exception as e:
            return f'❌ Maaf, terjadi kesalahan saat memproses jawaban: {str(e)}'


print('✅ Cell 8: ResponseGenerator siap (v3 — BUG 2 ordering fixed + sitasi lokal)')


✅ Cell 8: ResponseGenerator siap (v3 — BUG 2 ordering fixed + sitasi lokal)


## 🎯 Cell 9: Main Orchestrator — MovieRAGChatbot

> **Pipeline lengkap:**
> ```
> chat() ─→ guardrail ─→ router ─→ search ─→ [fallback] ─→ generate ─→ memory
> ```

In [21]:
from groq import Groq


class MovieRAGChatbot:
    """
    Orkestrator utama yang menggabungkan semua modul.

    Pipeline per pesan:
    1. Guardrail check      → tolak jika bukan topik film
    2. Query routing (LLM)  → ekstrak intent + explicit_title
    3. resolve_query_mode() → satu titik keputusan: cache/db/external
    4. Eksekusi mode        → ambil docs dari sumber yang tepat
    5. [BUG 3 FIX] db_title verification → cek title match, bukan sembarang film
    6. Fallback external    → jika DB kosong dan butuh sumber luar
    7. Response generation  → LLM + context + history
    8. Memory update        → simpan ke chat history
    """

    def __init__(
        self,
        vsm: VectorStoreManager,
        groq_api_key: str = GROQ_API_KEY,
        tmdb_api_key: str = TMDB_API_KEY,
        omdb_api_key: str = OMDB_API_KEY,
        verbose: bool = True
    ):
        self.verbose = verbose
        self.llm = Groq(api_key=groq_api_key)

        self.memory        = ChatMemory(max_turns=MEMORY_WINDOW_TURNS)
        self.guardrail     = TopicGuardrail(self.llm)
        self.router        = QueryRouter(self.llm)
        self.search_engine = MovieSearchEngine(vsm)
        self.external_api  = ExternalMovieAPI(tmdb_api_key, omdb_api_key)
        self.response_gen  = ResponseGenerator(self.llm)

        self._log('🎬 MovieRAGChatbot berhasil diinisialisasi!')
        self._log(f'   LLM         : {LLM_MODEL} via Groq')
        self._log(f'   Embedding   : {EMBEDDING_MODEL} (lokal)')
        self._log(f'   Memory      : {MEMORY_WINDOW_TURNS} turns')
        self._log(f'   External API: {"TMDB✅" if tmdb_api_key else "TMDB❌"} | {"OMDb✅" if omdb_api_key else "OMDb❌"}')

    def _log(self, msg: str) -> None:
        if self.verbose:
            print(msg)

    # ── Main Chat Method ──────────────────────────────────────
    def chat(self, user_input: str) -> str:
        user_input = user_input.strip()
        if not user_input:
            return '❓ Silakan ketikkan pertanyaan Anda.'

        self._log(f'\n{"─"*55}')
        self._log(f'👤 User: {user_input}')
        self._log(f'{"─"*55}')

        history_ctx = self.memory.get_recent_context(n_turns=3)

        # ── STEP 1: Guardrail ──────────────────────────────────
        self._log('[1/5] 🛡️  Guardrail check...')
        is_ok, rejection = self.guardrail.check(user_input, history_ctx)
        if not is_ok:
            self._log('       → DITOLAK (off-topic)')
            self.memory.add_user(user_input)
            self.memory.add_assistant(rejection)
            return rejection
        self._log('       → OK')

        # ── STEP 2: Query Routing (LLM) ───────────────────────
        self._log('[2/5] 🧭  Query routing...')
        intent = self.router.route(user_input, history_ctx)
        self._log(f'       → {intent.summary()}')

        # ── STEP 2.5: Resolve mode ────────────────────────────
        last_titles     = self.memory.last_titles
        last_docs       = self.memory.get_last_docs()
        last_docs_count = len(last_docs)

        mode = self.router.resolve_query_mode(
            query=user_input,
            intent=intent,
            last_titles=last_titles,
            last_docs_count=last_docs_count,
        )
        self._log(f'[2.5] 🔀  Mode: {mode}')

        # ── STEP 3: Eksekusi berdasarkan mode ─────────────────
        local_docs: List[Document] = []
        update_cache = False

        if mode == 'cache_index':
            ref_idx = self.router.extract_referenced_index(user_input)
            local_docs = [last_docs[ref_idx]]
            self._log(f'[3/5] 📚  Cache index [{ref_idx}]: "{last_titles[ref_idx]}"')

        elif mode == 'cache_title':
            lookup_key = intent.explicit_title if intent.explicit_title else user_input
            title_match = self.router.find_title_in_cache(lookup_key, last_titles)
            local_docs = [last_docs[title_match]]
            self._log(f'[3/5] 📚  Cache title: "{last_titles[title_match]}"')

        elif mode == 'cache_follow':
            local_docs = last_docs
            self._log(f'[3/5] 📚  Cache follow: {len(local_docs)} film dari list sebelumnya')

        elif mode == 'db_title':
            # ── [BUG 3 FIX] ─────────────────────────────────────────
            # Masalah lama: semantic_search limit=1 SELALU return 1 film,
            # bahkan jika film yang ditemukan TIDAK sesuai judul yang dicari.
            # Ini menyebabkan bot memberikan film acak (The Departed untuk The Conjuring, dst.)
            #
            # Fix: setelah search, VERIFIKASI apakah found_title cocok dengan explicit_title.
            # Gunakan word-intersection (abaikan stop words dan kata <= 2 karakter).
            # Jika TIDAK cocok → local_docs=[] → trigger fallback ke API eksternal.
            # ─────────────────────────────────────────────────────────
            title_query = intent.explicit_title
            self._log(f'[3/5] 📚  DB title search: "{title_query}"')

            # Cari beberapa kandidat untuk verifikasi
            title_intent = QueryIntent(
                strategy='semantic_search',
                semantic_query=f'movie film titled {title_query} {title_query}',
                limit=3,    # Ambil 3 kandidat, verifikasi 1 per 1
                raw_query=title_query,
            )
            candidates = self.search_engine.search(title_intent)

            # Verifikasi: cari kandidat yang title-nya benar-benar cocok
            local_docs = []
            for doc in candidates:
                found_title = doc.metadata.get('title', '')
                if self.router.verify_title_match(title_query, found_title):
                    local_docs = [doc]
                    self._log(f'       → 1 film cocok: "{found_title}"')
                    break

            if not local_docs:
                self._log(
                    f'       → 0 film cocok '
                    f'(judul "{title_query}" tidak ada di DB — akan coba fallback)'
                )
            # db_title TIDAK timpa cache list rekomendasi utama

        elif mode == 'similarity':
            # ── [BUG 1 FIX + BUG 5 FIX] ───────────────────────────
            # Masalah lama: mode cache_title dipilih sebelum similarity,
            # sehingga "film mirip X" hanya dapat X itu sendiri.
            #
            # Fix di resolve_query_mode: similarity dicek SEBELUM cache_title.
            # Fix di sini: filter source film dari hasil + UPDATE cache.
            # ─────────────────────────────────────────────────────────
            self._log(f'[3/5] 📚  Similarity search (semantic baru — cari film LAIN yang mirip)...')
            raw_results = self.search_engine.search(intent)

            # Filter: hilangkan film sumber dari hasil similarity
            if intent.explicit_title:
                source_words = {
                    w for w in intent.explicit_title.lower().split()
                    if len(w) > 3 and w not in self.router.TITLE_STOP_WORDS
                }
                local_docs = [
                    d for d in raw_results
                    if not any(
                        w in d.metadata.get('title', '').lower()
                        for w in source_words
                    )
                ] if source_words else raw_results
            else:
                local_docs = raw_results

            self._log(f'       → {len(local_docs)} film ditemukan (sumber dihapus dari hasil)')
            # [BUG 5 FIX] Update cache agar user bisa refer ke hasil similarity ("nomor 2 tadi")
            update_cache = True

        elif mode == 'external':
            self._log('[3/5] 📚  Skip local search (needs_external=True)')

        else:  # 'db_search' — pencarian reguler
            self._log('[3/5] 📚  DB search...')
            local_docs = self.search_engine.search(intent)
            self._log(f'       → {len(local_docs)} film ditemukan')
            update_cache = True

        # Update cache hanya untuk hasil search reguler & similarity
        if update_cache and local_docs:
            self.memory.save_results(local_docs)
            self._log(f'       → Cache diperbarui ({len(local_docs)} film)')

        # ── STEP 4: Fallback External API ──────────────────────
        external_data = None

        # Kondisi fallback: mode butuh eksternal ATAU hasil DB kosong (termasuk gagal verifikasi judul)
        should_fallback = (
            mode == 'external'
            or (mode == 'db_title' and len(local_docs) == 0)
            or (mode == 'db_search' and len(local_docs) == 0)
        )

        if should_fallback and self.external_api.is_available():
            self._log('[4/5] 🌐  Fetching external API...')
            ext_query = intent.explicit_title or intent.semantic_query or user_input
            external_data = self.external_api.search(ext_query, intent)
            n_ext = len(external_data[0]) if external_data else 0
            src   = external_data[1] if external_data else '-'
            self._log(f'       → {n_ext} film dari {src}')
        elif should_fallback:
            self._log('[4/5] 🌐  External API tidak tersedia (API key belum diset)')
        else:
            self._log('[4/5] 🌐  Fallback tidak diperlukan')

        # ── STEP 5: Generate Response ──────────────────────────
        self._log('[5/5] 💬  Generating response...')
        response = self.response_gen.generate(
            query=user_input,
            local_docs=local_docs,
            external_data=external_data,
            history=self.memory.get_history(),
            intent=intent,
            query_mode=mode,
        )

        self.memory.add_user(user_input)
        self.memory.add_assistant(response)
        self._log(f'   Memory: {len(self.memory)} turns tersimpan')

        return response

    # ── Utility Methods ───────────────────────────────────────
    def reset(self) -> None:
        self.memory.clear()
        self._log('🔄 Memory percakapan direset')

    def get_history(self) -> List[Dict]:
        return self.memory.get_history()


print('✅ Cell 9: MovieRAGChatbot siap (v3 — BUG 1,3,5 fixed)')


✅ Cell 9: MovieRAGChatbot siap (v3 — BUG 1,3,5 fixed)


## 🚀 Cell 10: Inisialisasi Sistem

> Jalankan cell ini sekali. Jika vector store sudah ada, akan di-load otomatis.

In [22]:
# ============================================================
# CELL 10 — INISIALISASI SISTEM
# ============================================================
# Jalankan ini sekali sebelum chatting

# Step 1: Load data
print('📂 Loading dataset...')
loader = MovieDataLoader()
df = loader.load(MOVIE_CSV_PATH)
docs = loader.to_documents(df)
print(f'   {len(docs)} dokumen siap untuk di-embed\n')

# Step 2: Vector store (auto build atau load)
# Tidak perlu api_key — embedding berjalan lokal!
print('🗄️ Menyiapkan Vector Store...')
vsm = VectorStoreManager(persist_dir=CHROMA_PERSIST_DIR)
vsm.build_or_load(docs)
print()

# Step 3: Inisialisasi chatbot
print('🤖 Menginisialisasi MovieRAGChatbot...')
chatbot = MovieRAGChatbot(
    vsm=vsm,
    groq_api_key=GROQ_API_KEY,
    # tmdb_api_key=TMDB_API_KEY,
    omdb_api_key=OMDB_API_KEY,
    verbose=True
)
print()
print('🎬 Sistem siap! Gunakan chatbot.chat("pertanyaan") untuk memulai.')


📂 Loading dataset...
📊 Dataset: 9988 film | Tahun: 1903–2023
   Kolom tersedia: ['title', 'year', 'rating', 'genre', 'overview', 'crew', 'orig_title', 'status', 'language', 'budget', 'revenue', 'country', 'cast']
   9988 dokumen siap untuk di-embed

🗄️ Menyiapkan Vector Store...
⏳ Memuat embedding model (download sekali ~30MB)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model siap: BAAI/bge-small-en-v1.5
✅ Vector store dimuat: 9988 dokumen dari /content/drive/MyDrive/PKA1/chroma_movie_db
💡 Tip: set FORCE_REBUILD=True untuk rebuild ulang

🤖 Menginisialisasi MovieRAGChatbot...
🎬 MovieRAGChatbot berhasil diinisialisasi!
   LLM         : llama-3.3-70b-versatile via Groq
   Embedding   : BAAI/bge-small-en-v1.5 (lokal)
   Memory      : 10 turns
   External API: TMDB❌ | OMDb✅

🎬 Sistem siap! Gunakan chatbot.chat("pertanyaan") untuk memulai.


## 🎭 Cell 11: Demo & Testing

> Test berbagai skenario: guardrail, metadata filter, semantic, multi-turn, fallback

In [25]:
# ============================================================
# CELL 11 — DEMO: TEST SKENARIO
# ============================================================

def demo_chat(chatbot, questions, section_title=''):
    """Helper untuk demo beberapa pertanyaan sekaligus."""
    if section_title:
        print(f'\n{"="*60}')
        print(f'  {section_title}')
        print(f'{"="*60}')
    for q in questions:
        resp = chatbot.chat(q)
        print(f'\n🤖 CineBot:\n{resp}')
        print('─' * 55)


# ── Test 1: Guardrail (harus ditolak) ─────────────────────────
demo_chat(chatbot, [
    'Berikan saya resep nasi goreng yang enak!',
    'Siapa presiden Indonesia saat ini?',
], section_title='TEST 1: GUARDRAIL — Harus Ditolak')

# ── Test 2: Metadata Filter ────────────────────────────────────
demo_chat(chatbot, [
    'Rekomendasikan film action dengan rating di atas 8',
    'Film horror terbaik tahun 2010 sampai 2020',
], section_title='TEST 2: METADATA FILTER')

# ── Test 3: Semantic Search ────────────────────────────────────
demo_chat(chatbot, [
    'Saya mau nonton film yang mirip Interstellar, ada saran?',
    'Tampilkan top 10 film dari hasil pencarian tadi',  # Multi-turn: 'tadi'
], section_title='TEST 3: SEMANTIC + MULTI-TURN')

# ── Test 4: Dynamic Output Limit ──────────────────────────────
demo_chat(chatbot, [
    'Kasih saya 3 film comedy terbaik saja',
], section_title='TEST 4: DYNAMIC OUTPUT LIMIT')

# ── Test 5: Fallback (Film > 2023) ────────────────────────────
demo_chat(chatbot, [
    'Berikan saya info tentang film Salmokji',
], section_title='TEST 5: FALLBACK EXTERNAL API')


  TEST 1: GUARDRAIL — Harus Ditolak

───────────────────────────────────────────────────────
👤 User: Berikan saya resep nasi goreng yang enak!
───────────────────────────────────────────────────────
[1/5] 🛡️  Guardrail check...
       → DITOLAK (off-topic)

🤖 CineBot:
🎬 Maaf, saya hanya bisa membantu tentang **film dan sinema**.

Sepertinya pertanyaan Anda di luar topik tersebut. Saya siap membantu dengan:

• 🎭 **Rekomendasi film** — berdasarkan genre, mood, atau vibe
• 🔍 **Info film** — plot, cast, rating, sutradara
• ⭐ **Perbandingan** — mana film yang lebih bagus?
• 🗓️ **Film by era** — film terbaik tahun tertentu

Ada film apa yang ingin Anda tanyakan? 😊
───────────────────────────────────────────────────────

───────────────────────────────────────────────────────
👤 User: Siapa presiden Indonesia saat ini?
───────────────────────────────────────────────────────
[1/5] 🛡️  Guardrail check...
       → DITOLAK (off-topic)

🤖 CineBot:
🎬 Maaf, saya hanya bisa membantu tentang **film da

## 💬 Cell 12: Mode Chat Interaktif

In [13]:
# ============================================================
# CELL 12 — INTERACTIVE CHAT (Jalankan jika ingin chat manual)
# ============================================================

def run_interactive(chatbot: MovieRAGChatbot):
    """Mode chat interaktif via terminal/Jupyter input."""
    print('\n' + '='*60)
    print('  🎬 CineBot — Movie RAG Chatbot (ketik quit untuk keluar)')
    print('='*60)
    print('  Perintah: quit / reset')
    print('='*60 + '\n')

    while True:
        try:
            user_input = input('👤 Anda: ').strip()
            if not user_input:
                continue
            if user_input.lower() in ('quit', 'exit', 'keluar'):
                print('\n👋 Sampai jumpa! Selamat menonton! 🍿')
                break
            if user_input.lower() == 'reset':
                chatbot.reset()
                print('🔄 Percakapan direset.\n')
                continue

            response = chatbot.chat(user_input)
            print(f'\n🤖 CineBot:\n{response}\n')
            print('─' * 55)

        except KeyboardInterrupt:
            print('\n\n👋 Sampai jumpa!')
            break


# Uncomment baris di bawah untuk memulai chat interaktif:
chatbot.reset()   # Reset history dulu
run_interactive(chatbot)

🔄 Memory percakapan direset

  🎬 CineBot — Movie RAG Chatbot (ketik quit untuk keluar)
  Perintah: quit / reset

👤 Anda: quit

👋 Sampai jumpa! Selamat menonton! 🍿
